In [1]:
from sklearn.datasets import make_moons
import torch
import numpy as np

X, Y = make_moons(
    n_samples=1000,
    noise=0.2,
    random_state=42
)

W1 = np.random.normal(0, 0.5, size=(2, 3))
b1 = np.zeros((1, 3))

W2 = np.random.normal(0, 0.5, size=(3, 1))
b2 = np.zeros((1, 1))

In [2]:
# numpy实现网络结构
# 结构：X(2)->Linear(3)->ReLU->Linear(1)->Sigmoid->BCE
def model_forward(x, y, W1, b1, W2, b2):
    # x:(1, 2), y:(1, 1)
    # W1:(2, 3),b1:(1, 3), Z1:(1, 3)
    Z1 = x @ W1 + b1
    
    # A1:(1, 3)
    A1 = np.maximum(0, Z1)
    
    # W2:(3, 1),b2:(1, 1)
    Z2 = A1 @ W2 + b2
    
    # P:(1, 1)
    P = 1 / (1 + np.exp(-Z2))
    
    # L:(1, 1)
    L = -(y * np.log(P) + (1 - y) * np.log(1 - P))

    return Z1, A1, Z2, P, L

In [9]:
# 选单个样本测试前向传播
x = X[:1]
y = Y[:1].reshape(1, 1)

z1, a1, z2, p, l = model_forward(x, y, W1, b1, W2, b2)
print("单样本前向传播测试：")
print("z1:", z1)
print("a1:", a1)
print("z2:", z2)
print("p:", p)
print("l:", l)

单样本前向传播测试：
z1: [[0.01592962 0.50967459 0.21712788]]
a1: [[0.01592962 0.50967459 0.21712788]]
z2: [[-0.0333152]]
p: [[0.49167197]]
l: [[0.70994351]]


# 计算图：
![计算图2](./images/计算图2.jpg)

In [10]:
# 手动链式法则求梯度
# grad_l:(1, 1)
grad_l = 1

# grad_p:(1, 1)
grad_p = -y / p + (1 - y) / (1 - p)

# grad_z2:(1, 1)
grad_z2 = grad_p * p * (1 - p)

# grad_b2:(1, 1)
grad_b2 = grad_z2

# grad_W2:(3, 1)
grad_W2 = a1.T @ grad_z2 

# grad_a1:(1, 3)
grad_a1 = grad_z2 @ W2.T

# grad_z1:(1, 3)
grad_z1 = grad_a1 * (z1 > 0)

# grad_b1:(1, 3)
grad_b1 = grad_z1

# grad_W1:(2, 3)
grad_W1 = x.T @ grad_z1

print("grad_W1:", grad_W1)
print("grad_b1:", grad_b1)
print("grad_W2:", grad_W2)
print("grad_b2:", grad_b2)

grad_W1: [[-0.04604662  0.01290205 -0.0356169 ]
 [ 0.21451853 -0.06010712  0.16592933]]
grad_b1: [[ 0.41235822 -0.11554089  0.31895763]]
grad_W2: [[-0.00809747]
 [-0.25908188]
 [-0.11037219]]
grad_b2: [[-0.50832803]]


In [12]:
# Pytorch自动求导验证
x_t = torch.tensor(x, dtype=torch.float64)
y_t = torch.tensor(y, dtype=torch.float64)

W1_t = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
b1_t = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
W2_t = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
b2_t = torch.tensor(b2, dtype=torch.float64, requires_grad=True)

z1_t = x_t @ W1_t + b1_t
a1_t = torch.relu(z1_t)
z2_t = a1_t @ W2_t + b2_t
p_t = torch.sigmoid(z2_t)

loss_t = torch.nn.functional.binary_cross_entropy(p_t, y_t)

loss_t.backward()

# W1
print("W1")
print("Shape:", grad_W1.shape)
print("NumPy:", grad_W1)
print("PyTorch:", W1_t.grad.numpy())
print("最大绝对误差:", np.max(np.abs(grad_W1 - W1_t.grad.numpy())))

# b1
print("\nb1")
print("Shape:", grad_b1.shape)
print("NumPy:", grad_b1)
print("PyTorch:", b1_t.grad.numpy())
print("最大绝对误差:", np.max(np.abs(grad_b1 - b1_t.grad.numpy())))

# W2
print("\nW2")
print("Shape:", grad_W2.shape)
print("NumPy:", grad_W2)
print("PyTorch:", W2_t.grad.numpy())
print("最大绝对误差:", np.max(np.abs(grad_W2 - W2_t.grad.numpy())))

# b2
print("\nb2")
print("Shape:", grad_b2.shape)
print("NumPy:", grad_b2)
print("PyTorch:", b2_t.grad.numpy())
print("最大绝对误差:", np.max(np.abs(grad_b2 - b2_t.grad.numpy())))

W1
Shape: (2, 3)
NumPy: [[-0.04604662  0.01290205 -0.0356169 ]
 [ 0.21451853 -0.06010712  0.16592933]]
PyTorch: [[-0.04604662  0.01290205 -0.0356169 ]
 [ 0.21451853 -0.06010712  0.16592933]]
最大绝对误差: 0.0

b1
Shape: (1, 3)
NumPy: [[ 0.41235822 -0.11554089  0.31895763]]
PyTorch: [[ 0.41235822 -0.11554089  0.31895763]]
最大绝对误差: 0.0

W2
Shape: (3, 1)
NumPy: [[-0.00809747]
 [-0.25908188]
 [-0.11037219]]
PyTorch: [[-0.00809747]
 [-0.25908188]
 [-0.11037219]]
最大绝对误差: 1.734723475976807e-18

b2
Shape: (1, 1)
NumPy: [[-0.50832803]]
PyTorch: [[-0.50832803]]
最大绝对误差: 0.0


# NumPy手算梯度与PyTorch自动求导梯度完全一致

In [15]:
# batch前向传播
# x:(4, 2),y:(4, 1)
x_batch = X[:4]
y_batch = Y[:4].reshape(4, 1)

# z1:(4, 3)
z1_batch = x_batch @ W1 + b1

# a1:(4, 3)
a1_batch = np.maximum(0, z1_batch)

# z2:(4, 1)
z2_batch = a1_batch @ W2 + b2

# p:(4, 1)
p_batch = 1 / (1 + np.exp(-z2_batch))

# l:标量
l_batch = -np.mean(
    y_batch * np.log(p_batch)
    + (1 - y_batch) * np.log(1 - p_batch)
)

print("z1:", z1_batch)
print("a1:", a1_batch)
print("z2:", z2_batch)
print("p:", p_batch)
print("l:", l_batch)

z1: [[ 0.01592962  0.50967459  0.21712788]
 [ 0.17528587 -0.57034552  0.44003844]
 [ 0.13746373 -0.18532892  0.42772704]
 [-0.01805538 -0.53928034 -0.23398614]]
a1: [[0.01592962 0.50967459 0.21712788]
 [0.17528587 0.         0.44003844]
 [0.13746373 0.         0.42772704]
 [0.         0.         0.        ]]
z2: [[-0.0333152 ]
 [-0.41830113]
 [-0.37989466]
 [ 0.        ]]
p: [[0.49167197]
 [0.39692335]
 [0.4061523 ]
 [0.5       ]]
l: 0.8070324623804072


In [16]:
# batch反向传播
batch_size = x_batch.shape[0]

# grad_l:标量
grad_l_batch = 1

# grad_p:(4, 1)
grad_p_batch = (
    -y_batch / p_batch
    + (1 - y_batch) / (1 - p_batch)
) / batch_size

# grad_z2:(4, 1)
grad_z2_batch = grad_p_batch * p_batch * (1 - p_batch)

# grad_b2:(1, 1)
grad_b2_batch = np.sum(grad_z2_batch, axis=0, keepdims=True)

# grad_W2:(3, 1)
grad_W2_batch = a1_batch.T @ grad_z2_batch

# grad_a1:(4, 3)
grad_a1_batch = grad_z2_batch @ W2.T

# grad_z1:(4, 3)
grad_z1_batch = grad_a1_batch * (z1_batch > 0)

# grad_b1:(1, 3)
grad_b1_batch = np.sum(grad_z1_batch, axis=0, keepdims=True)

# grad_W1:(2, 3)
grad_W1_batch = x_batch.T @ grad_z1_batch

print("grad_W1:", grad_W1_batch)
print("grad_b1:", grad_b1_batch)
print("grad_W2:", grad_W2_batch)
print("grad_b2:", grad_b2_batch)

grad_W1: [[ 0.22405125  0.00322551  0.17330286]
 [ 0.01035278 -0.01502678  0.00800784]]
grad_b1: [[ 0.34582729 -0.02888522  0.2674962 ]]
grad_W2: [[-0.0488602 ]
 [-0.06477047]
 [-0.15743845]]
grad_b2: [[-0.55131309]]


In [17]:
# PyTorch自动求导验证
x_t_batch = torch.tensor(x_batch, dtype=torch.float64)
y_t_batch = torch.tensor(y_batch, dtype=torch.float64)

W1_t_batch = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
b1_t_batch = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
W2_t_batch = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
b2_t_batch = torch.tensor(b2, dtype=torch.float64, requires_grad=True)

z1_t_batch = x_t_batch @ W1_t_batch + b1_t_batch
a1_t_batch = torch.relu(z1_t_batch)
z2_t_batch = a1_t_batch @ W2_t_batch + b2_t_batch
p_t_batch = torch.sigmoid(z2_t_batch)

loss_t_batch = torch.nn.functional.binary_cross_entropy(
    p_t_batch, y_t_batch
)

loss_t_batch.backward()

# W1
print("W1")
print("Shape:", grad_W1_batch.shape)
print("NumPy:", grad_W1_batch)
print("PyTorch:", W1_t_batch.grad.numpy())
print("最大绝对误差:", np.max(np.abs(
    grad_W1_batch - W1_t_batch.grad.numpy()
)))

# b1
print("\nb1")
print("Shape:", grad_b1_batch.shape)
print("NumPy:", grad_b1_batch)
print("PyTorch:", b1_t_batch.grad.numpy())
print("最大绝对误差:", np.max(np.abs(
    grad_b1_batch - b1_t_batch.grad.numpy()
)))

# W2
print("\nW2")
print("Shape:", grad_W2_batch.shape)
print("NumPy:", grad_W2_batch)
print("PyTorch:", W2_t_batch.grad.numpy())
print("最大绝对误差:", np.max(np.abs(
    grad_W2_batch - W2_t_batch.grad.numpy()
)))

# b2
print("\nb2")
print("Shape:", grad_b2_batch.shape)
print("NumPy:", grad_b2_batch)
print("PyTorch:", b2_t_batch.grad.numpy())
print("最大绝对误差:", np.max(np.abs(
    grad_b2_batch - b2_t_batch.grad.numpy()
)))

W1
Shape: (2, 3)
NumPy: [[ 0.22405125  0.00322551  0.17330286]
 [ 0.01035278 -0.01502678  0.00800784]]
PyTorch: [[ 0.22405125  0.00322551  0.17330286]
 [ 0.01035278 -0.01502678  0.00800784]]
最大绝对误差: 2.7755575615628914e-17

b1
Shape: (1, 3)
NumPy: [[ 0.34582729 -0.02888522  0.2674962 ]]
PyTorch: [[ 0.34582729 -0.02888522  0.2674962 ]]
最大绝对误差: 0.0

W2
Shape: (3, 1)
NumPy: [[-0.0488602 ]
 [-0.06477047]
 [-0.15743845]]
PyTorch: [[-0.0488602 ]
 [-0.06477047]
 [-0.15743845]]
最大绝对误差: 0.0

b2
Shape: (1, 1)
NumPy: [[-0.55131309]]
PyTorch: [[-0.55131309]]
最大绝对误差: 0.0


# NumPy手算梯度与PyTorch自动求导梯度完全一致